# 03d · Gerçek NAS: final mimari ve Nano, sıfırdan 300 epoch × 3 tohum (`nas_final1`)

**Ne yapar:** 03c'nin önceden kayıtlı kuralla seçtiği **finalist** ile **orijinal YOLOX-Nano@416**'yı aynı koşullarda, **sıfırdan 300 epoch** (YOLOX makalesinin tarifi), **üçer tohumla** eğitir (Karar 6: tohumlar baştan planlı, ilk sonuca bağlı değil). Eşli tasarım: tohum s her iki mimaride aynı (başlatma tohumu = veri/artırma tohumu = s).

**İddia sınırı:** 3 tekrar sabit mimarinin eğitim değişkenliğini ölçer; aramanın farklı tohumlarda aynı kalitede mimari bulduğunu göstermez. 60 epoch'taki −1,0 AP koşulu 300 epoch'u garanti etmez; nihai hedef burada, aynı nihai kümede ortalama AP farkı ve tekrar dağılımıyla raporlanır. Karşılanmazsa açıklanır; B/test-dev'e bakıp aday değiştirilmez.

**Tarif:** YOLOX-Nano sıfırdan, değiştirilmemiş: 300 epoch, ısınma 5, son 15 (+1) epoch mosaic'siz, lr 0,01/64, batch 64 (resmî 128/8 GPU; görsel başına lr aynı), fp16, EMA, mosaic 0,5, mixup kapalı, çok ölçek. Resmî 25,8 batch 128/8 GPU ile eğitildi → karşılaştırma bu koşulun Nano'su ile yapılır (resmî değer yalnız bağlam).

**Değerlendirme:** geliştirme için val2017-A (her koşu sonunda, kendi çözünürlüğünde, fp32). **Nihai değerlendirme bu notebook'ta yapılmaz:** NAS + budama + nicemleme yöntem kararları bitince ayrı mühürlü değerlendirme (test-dev, gönderim kabulü doğrulanırsa; değilse val2017-B).

**Kesintiye dayanıklılık:** koşu başına doğrulanmış depo, 300 iterasyonda bir + epoch sonu; 100/200/300. epoch kayıtları sabit (öğrenme eğrisi). Hücre 9 yeniden çalıştırılınca kaldığı yerden.

**Süre/maliyet (tahmin):** 300 × 886 sn ≈ 74 sa (Nano), finalist mimariye göre ±%10 → 6 koşu ≈ 450–490 sa ≈ 690–750 CU, tek GPU'da ≈ 19–20 gün. Sıra: (Nano s0, final s0), (Nano s1, final s1), (Nano s2, final s2).

**Kayıtlar** (`Drive edge_ai/results/nas_final1/`): `manifest.json` · `runs/<mimari>_s<tohum>/{status.json, train_log.txt, eval_A_log.txt, ckpt/}` · `results.csv` · `nas_final1_summary.json`.

In [ ]:
# 1 · Ortam ve sürüm kaydı
import json, platform, subprocess, sys, datetime, os

def sh(cmd):
    try:
        return subprocess.run(cmd, shell=True, capture_output=True, text=True).stdout.strip()
    except Exception as e:
        return f"ERR {e}"

env = {
    "timestamp_utc": datetime.datetime.now(datetime.UTC).isoformat(timespec="seconds"),
    "python": sys.version.split()[0],
    "platform": platform.platform(),
    "gpu": sh("nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader") or "none",
    "cuda_nvcc": sh("nvcc --version | tail -1"),
    "cpu_count": os.cpu_count(),
    "ram_gb": round(os.sysconf("SC_PAGE_SIZE") * os.sysconf("SC_PHYS_PAGES") / 1e9, 1),
    "disk_free_content": sh("df -h /content | tail -1"),
}
try:
    import torch, torchvision
    env.update(torch=torch.__version__, torchvision=torchvision.__version__,
               torch_cuda=torch.version.cuda, cuda_available=torch.cuda.is_available())
except ImportError:
    env["torch"] = "not installed"
print(json.dumps(env, indent=2))

In [ ]:
# 2 · Kod: depo + YOLOX fork sabit commit (exp ve betik depoda: exps/nas, scripts)
REPO_URL = "https://github.com/Bey3fendi/edge-ai-object-detection.git"
YOLOX_COMMIT = "3ec6263"
WS = "/content/edge_ai_workspace"

import os, subprocess
if not os.path.isdir(WS):
    subprocess.run(["git", "clone", "--recurse-submodules", REPO_URL, WS], check=True)
else:  # runtime aynı kaldıysa depodaki son exp/betik sürümünü al
    subprocess.run(["git", "-C", WS, "pull", "-q", "--ff-only"], check=True)
subprocess.run(["git", "-C", f"{WS}/YOLOX", "fetch", "--all", "-q"], check=True)
subprocess.run(["git", "-C", f"{WS}/YOLOX", "checkout", "-q", YOLOX_COMMIT], check=True)

env["repo_commit"] = sh(f"git -C {WS} rev-parse --short HEAD")
env["yolox_commit"] = sh(f"git -C {WS}/YOLOX rev-parse --short HEAD")
env["yolox_dirty"] = bool(sh(f"git -C {WS}/YOLOX status --porcelain"))
print({k: env[k] for k in ("repo_commit", "yolox_commit", "yolox_dirty")})
assert env["yolox_commit"].startswith(YOLOX_COMMIT)
env["repo_dirty"] = bool(sh(f"git -C {WS} status --porcelain"))
assert not env["repo_dirty"], "depo kirli: kayıtlı commit koşulan kodu temsil etmez"

In [ ]:
# 3 · YOLOX bağımlılıkları
# Not: `pip install -e .` (setup.py develop) güncel setuptools + py3.13'te kırılıyor; YOLOX saf Python
# olduğu için kurulum yerine PYTHONPATH kullanılır. onnx-simplifier eğitimde gerekmez → atlandı.
# onnx + onnxruntime: check_supernet.py'nin ONNX opset 13 eşdeğerlik denetimi için (hücre 7).
!pip install -q loguru tabulate thop ninja pycocotools psutil tensorboard opencv-python-headless onnx onnxruntime

import sys, os
YOLOX_DIR = f"{WS}/YOLOX"
if YOLOX_DIR not in sys.path:
    sys.path.insert(0, YOLOX_DIR)
os.environ["PYTHONPATH"] = YOLOX_DIR + ":" + os.environ.get("PYTHONPATH", "")

import importlib.metadata as md
for p in ["torch", "torchvision", "numpy", "opencv-python-headless", "pycocotools", "loguru", "thop", "onnx", "onnxruntime"]:
    try:
        env[f"pkg_{p}"] = md.version(p)
    except md.PackageNotFoundError:
        env[f"pkg_{p}"] = None
import yolox
from yolox.exp import get_exp
exp = get_exp(None, "yolox-nano")
print("yolox import OK:", yolox.__file__)
print({k: v for k, v in env.items() if k.startswith("pkg_")})
SCRIPTS = f"{WS}/scripts"
if SCRIPTS not in sys.path:
    sys.path.insert(0, SCRIPTS)
NAS_EXPS = f"{WS}/exps/nas"
if NAS_EXPS not in sys.path:
    sys.path.insert(0, NAS_EXPS)
import importlib, nas_search as ns, nas_data as nd, supernet as sn, supernet_search as ss
for _m in (ns, nd, sn, ss):
    importlib.reload(_m)
EXP_FILE = f"{WS}/exps/nas/scratch_subnet.py"
import rank_select as rs, nas_export as nx
importlib.reload(rs); importlib.reload(nx)

In [ ]:
# 4 · Drive + ağırlık (SHA256)
# İlk mount tarayıcıda kullanıcı onayı ister; bağlıysa tekrar denemez.
env.pop("pkg_yolox", None)
if not os.path.ismount("/content/drive"):
    from google.colab import drive
    drive.mount("/content/drive")

DRIVE_ROOT = "/content/drive/MyDrive/edge_ai"
DRIVE_DATA = f"{DRIVE_ROOT}/data"
DRIVE_WEIGHTS = f"{DRIVE_ROOT}/weights"
ROUND = "nas_final1"   # gerçek NAS, final + Nano sıfırdan 300 ep × 3 tohum (Karar 6)
SUPER = f"{DRIVE_ROOT}/results/nas_super1"; RANK = f"{DRIVE_ROOT}/results/nas_rank1"
DRIVE_RESULTS = f"{DRIVE_ROOT}/results/{ROUND}"
for d in (DRIVE_DATA, DRIVE_WEIGHTS, DRIVE_RESULTS):
    os.makedirs(d, exist_ok=True)

import hashlib, urllib.request, shutil
WEIGHT_URL = "https://github.com/Megvii-BaseDetection/YOLOX/releases/download/0.1.1rc0/yolox_nano.pth"
w_drive = f"{DRIVE_WEIGHTS}/yolox_nano.pth"
if not os.path.exists(w_drive):
    urllib.request.urlretrieve(WEIGHT_URL, w_drive)
WEIGHTS = "/content/yolox_nano.pth"
shutil.copy(w_drive, WEIGHTS)

h = hashlib.sha256()
with open(WEIGHTS, "rb") as f:
    for chunk in iter(lambda: f.read(1 << 20), b""):
        h.update(chunk)
env["weights_url"] = WEIGHT_URL
env["weights_sha256"] = h.hexdigest()
env["weights_bytes"] = os.path.getsize(WEIGHTS)
print(env["weights_sha256"], env["weights_bytes"])

In [ ]:
# 5 · Veri: tam COCO train2017 + val2017 (Colab yereline)
# train2017 görüntüleri Drive'da yok: Drive'da `data/train2017.zip` varsa oradan, yoksa resmi COCO sunucusundan
# Colab yereline indirilir (≈18 GB). Drive kotası için zip varsayılan olarak Drive'a KAYDEDİLMEZ; her yeni
# runtime'da yeniden indirilir. Anotasyonlar ve val2017 P0'dan beri Drive'da.
# Kesinti testi (hücre 7) için 10k alt kümenin yalnız anotasyon json'u açılır (görselleri train2017'de zaten var).
import zipfile, shutil, time
COCO_LOCAL = "/content/datasets/COCO"
ANN_ZIP = f"{DRIVE_DATA}/annotations_trainval2017.zip"
VAL_ZIP = f"{DRIVE_DATA}/val2017.zip"
SUBSET_ZIP = f"{DRIVE_DATA}/coco_train2017_10k_stratified.zip"
TRAIN_ZIP_DRIVE = f"{DRIVE_DATA}/train2017.zip"
TRAIN_ZIP_URL = "http://images.cocodataset.org/zips/train2017.zip"
SAVE_TRAIN_ZIP_TO_DRIVE = False   # True: ilk indirmede Drive'a da kopyalar (≈18 GB kota)
N_TRAIN = 118_287
for z in (ANN_ZIP, VAL_ZIP, SUBSET_ZIP):
    assert os.path.exists(z), f"{z} yok: önce 00 pilot notebook'unun hücre 5 ve 9'u"
t0 = time.time()
os.makedirs(f"{COCO_LOCAL}/annotations", exist_ok=True)
with zipfile.ZipFile(ANN_ZIP) as z:
    for name in ("annotations/instances_train2017.json", "annotations/instances_val2017.json"):
        if not os.path.exists(f"{COCO_LOCAL}/{name}"):
            z.extract(name, COCO_LOCAL)
with zipfile.ZipFile(SUBSET_ZIP) as z:
    if not os.path.exists(f"{COCO_LOCAL}/annotations/instances_train2017_10k.json"):
        z.extract("annotations/instances_train2017_10k.json", COCO_LOCAL)
if not os.path.isdir(f"{COCO_LOCAL}/val2017"):
    shutil.copy(VAL_ZIP, "/content/val2017.zip")
    with zipfile.ZipFile("/content/val2017.zip") as z:
        z.extractall(COCO_LOCAL)

n_local = len(os.listdir(f"{COCO_LOCAL}/train2017")) if os.path.isdir(f"{COCO_LOCAL}/train2017") else 0
if n_local != N_TRAIN:
    local_zip = "/content/train2017.zip"
    if os.path.exists(TRAIN_ZIP_DRIVE):
        print("train2017.zip Drive'dan kopyalanıyor…", flush=True)
        shutil.copy(TRAIN_ZIP_DRIVE, local_zip)
        env["train_zip_source"] = "drive"
    else:
        print("train2017.zip resmi COCO sunucusundan indiriliyor (≈18 GB)…", flush=True)
        subprocess.run(["wget", "-q", "--show-progress", "--progress=dot:giga", "-O", local_zip + ".part",
                        TRAIN_ZIP_URL], check=True)
        os.replace(local_zip + ".part", local_zip)
        env["train_zip_source"] = TRAIN_ZIP_URL
    env["train_zip_bytes"] = os.path.getsize(local_zip)
    env["train_zip_sha256"] = ns.sha256(local_zip)
    if SAVE_TRAIN_ZIP_TO_DRIVE and not os.path.exists(TRAIN_ZIP_DRIVE):
        shutil.copy(local_zip, TRAIN_ZIP_DRIVE + ".tmp")
        os.replace(TRAIN_ZIP_DRIVE + ".tmp", TRAIN_ZIP_DRIVE)
    print("açılıyor…", flush=True)
    subprocess.run(["unzip", "-q", "-o", local_zip, "-d", COCO_LOCAL], check=True)
    os.remove(local_zip)
env["train_images"] = len(os.listdir(f"{COCO_LOCAL}/train2017"))
env["val_images"] = len(os.listdir(f"{COCO_LOCAL}/val2017"))
env["train_ann_sha256"] = ns.sha256(f"{COCO_LOCAL}/annotations/instances_train2017.json")
env["data_prep_s"] = round(time.time() - t0, 1)
env["disk_free_after_data"] = sh("df -h /content | tail -1")
print({k: env.get(k) for k in ("train_images", "val_images", "train_zip_source", "train_zip_sha256",
                               "train_ann_sha256", "data_prep_s", "disk_free_after_data")})
assert env["train_images"] == N_TRAIN and env["val_images"] == 5000

In [ ]:
# 6 · Veri ayrımları (03 ile birebir aynı olmalı)
SPLIT_SEED = 2026
sm_local = f"{COCO_LOCAL}/annotations/nas_splits_manifest.json"
if not os.path.exists(sm_local):
    r = subprocess.run(["python", f"{WS}/scripts/make_splits.py", "--coco", COCO_LOCAL, "--seed", str(SPLIT_SEED)],
                       capture_output=True, text=True)
    print(r.stdout[-1500:], r.stderr[-1500:]); assert r.returncode == 0
sm = json.load(open(sm_local))
sm_super = ns.read_json(f"{SUPER}/splits_manifest.json")
assert sm["ids_sha256"] == sm_super["ids_sha256"], "ayrım 03'tekinden farklı"
print("ayrım 03 ile aynı:", sm["ids_sha256"]["A"][:12])
# MÜHÜR: instances_val2017_B.json hiçbir yükleyiciye verilmez.

In [ ]:
# 7 · Finalist + koşu planı + öz-denetim (kesinti testi 03c'de GPU'da geçti; burada CPU testi tekrar)
import torch, re, time, signal
from ckpt_store import CheckpointStore
fin = ns.read_json(f"{RANK}/finalist.json")
assert fin, "03c finalist kararı yok"
print("03c kararı:", fin["verdict"], fin["finalist"])
assert fin["finalist"], "bu protokolde uygun aday bulunamadı → 300 ep finali koşulmaz (Karar 6), konuşulmalı"
NANO_ID = next(r for r in ns.read_json(f"{RANK}/manifest.json")["runs"] if r["group"] == "nano")["config_id"]
SEEDS = (0, 1, 2)
RUNS = []
for s in SEEDS:
    for tag, cid in (("nano", NANO_ID), ("final", fin["finalist"])):
        RUNS.append({"run_id": f"{tag}_s{s}", "arch": tag, "config_id": cid, "seed": s,
                     "res": sn.parse_config_id(cid)["res"]})
for r in RUNS:
    print(r)
rr = subprocess.run(["python", f"{WS}/scripts/check_trainer_cpu.py", "--exp", "scratch"], capture_output=True, text=True, cwd=WS)
print(rr.stdout[-300:]); assert "GEÇTİ" in rr.stdout.splitlines()[-1], rr.stderr[-3000:]
BATCH, FP16, CU_PER_HOUR = 64, True, 1.54
WORKERS = os.cpu_count()
OUT_ROOT = f"{WS}/YOLOX_outputs/{ROUND}"
MAX_EPOCH, WARMUP, NO_AUG, CKPT_EVERY = 300, 5, 15, 300     # YOLOX-Nano resmî sıfırdan tarifi
def recipe_opts(seed):
    return ["max_epoch", str(MAX_EPOCH), "warmup_epochs", str(WARMUP), "no_aug_epochs", str(NO_AUG),
            "nas_seed_init", str(seed), "nas_seed_data", str(seed)]
def run_train(opts, tag, env_extra=None, kill_at=None):
    cmd = ["python", "-u", "YOLOX/tools/train.py", "-f", EXP_FILE, "-d", "1", "-b", str(BATCH), "--fp16",
           "-expn", tag] + opts
    t0 = time.time()
    proc = subprocess.Popen(cmd, cwd=WS, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
                            env={**os.environ, **(env_extra or {})})
    lines, last = [], 0
    for line in proc.stdout:
        lines.append(line)
        m = re.search(r"epoch: (\d+)/\d+, iter: (\d+)/(\d+)", line)
        if (m and time.time() - last > 120) or re.search(r"devam:|\[test\]|Traceback|Error", line):
            print(f"[{tag} {time.time() - t0:6.0f}s] {line.strip()[-150:]}", flush=True); last = time.time()
        if kill_at is not None and m and (int(m.group(1)) - 1) * int(m.group(3)) + int(m.group(2)) >= kill_at:
            proc.send_signal(signal.SIGKILL); print(f"[{tag}] SIGKILL @ ≥{kill_at}", flush=True)
    proc.wait()
    return proc.returncode, "".join(lines), time.time() - t0



In [ ]:
# 8 · Dondurulmuş manifest
from yolox.exp import get_exp
for _m in ("scratch_subnet", "supernet", "nano_macro"):
    sys.modules.pop(_m, None)
_e = get_exp(EXP_FILE, None)
_e.merge(["nas_arch", NANO_ID] + recipe_opts(0))
RECIPE_KEYS = ("max_epoch", "warmup_epochs", "warmup_lr", "no_aug_epochs", "basic_lr_per_img", "min_lr_ratio",
               "scheduler", "weight_decay", "momentum", "ema", "input_size", "random_size", "mosaic_prob",
               "mosaic_scale", "enable_mixup", "mixup_prob", "hsv_prob", "flip_prob", "degrees", "translate", "shear",
               "train_ann", "val_ann", "nmsthre")
FINGERPRINT = ("round", "runs", "recipe", "eval_protocol", "finalist_sha256", "exp_file_sha256", "train_ann_sha256",
               "splits", "yolox_commit")
manifest = ns.ensure_manifest(f"{DRIVE_RESULTS}/manifest.json", {
    "round": ROUND, "notebook": "notebooks/03d_gercek_nas_final_300ep.ipynb", "runs": RUNS,
    "finalist_sha256": ns.sha256(f"{RANK}/finalist.json"), "finalist": fin["finalist"], "nano": NANO_ID,
    "recipe": {**{k: getattr(_e, k, None) for k in RECIPE_KEYS}, "batch": BATCH, "fp16": FP16, "init": "sıfırdan (YOLOX)",
               "seeds": list(SEEDS), "seed_design": "eşli: tohum s iki mimaride aynı; nas_seed_init = nas_seed_data = s",
               "official_reference": "resmî YOLOX-Nano 25,8 (batch 128, 8 GPU) yalnız bağlam"},
    "eval_protocol": {"development": "val2017 A (2500), son epoch EMA, fp32, kendi çözünürlüğü",
                      "final": "bu notebook'ta yok: mühürlü değerlendirme yöntem kararları bitince (test-dev / val2017 B)"},
    "claim_scope": "3 tekrar: sabit mimarinin eğitim değişkenliği; arama tekrarlanabilirliği değil",
    "exp_file_sha256": ns.sha256(EXP_FILE), "train_ann_sha256": env["train_ann_sha256"],
    "splits": {"ids_sha256": sm["ids_sha256"], "sealed": "val2017 B açılmaz"},
    "yolox_commit": env["yolox_commit"], "repo_commit_at_creation": env["repo_commit"], "cu_per_hour": CU_PER_HOUR,
    "env_at_creation": env}, keys=FINGERPRINT)
del _e
print(json.dumps({k: manifest[k] for k in ("round", "created_utc", "finalist", "nano")}, indent=1, default=str))

In [ ]:
# 9 · Eğitim + val-A değerlendirmesi (UZUN KOŞU ≈ 19–20 gün — Furkan'ın başlatma onayıyla). Kaldığı yerden.
# Sıra: (Nano s0, final s0), (Nano s1, final s1), (Nano s2, final s2); her koşu bitince val-A.
IPE = 118_287 // BATCH
def run_dir(r):
    return f"{DRIVE_RESULTS}/runs/{r['run_id']}"

def train_eval(r):
    st = ns.get_status(DRIVE_RESULTS, r["run_id"])
    if st["status"] == "evaluated":
        return st
    d = run_dir(r); os.makedirs(d, exist_ok=True)
    store = CheckpointStore(f"{d}/ckpt", f"{OUT_ROOT}/{r['run_id']}/ck_local", log=print)
    recs = store.records() if os.path.isdir(f"{d}/ckpt") else []
    p = recs[-1]["progress"] if recs else None
    if p is None or p < MAX_EPOCH * IPE:
        st = ns.set_status(DRIVE_RESULTS, r["run_id"], st, status="training", attempts=st["attempts"] + [
            {"start_utc": ns.now_utc(), "resume_from_progress": p, "gpu": env["gpu"], "repo_commit": env["repo_commit"]}])
        rc, lg, wall = run_train(["nas_arch", r["config_id"], "nas_ckpt_dir", f"{d}/ckpt", "nas_log_dir", d,
                                  "output_dir", OUT_ROOT, "data_num_workers", str(WORKERS),
                                  "nas_ckpt_every_iters", str(CKPT_EVERY), "nas_snapshot_epochs", "100,200"]
                                 + recipe_opts(r["seed"]),
                                 r["run_id"])
        st["attempts"][-1].update(end_utc=ns.now_utc(), wall_s=round(wall, 1), rc=rc)
        st = ns.set_status(DRIVE_RESULTS, r["run_id"], st)
        assert rc == 0, f"{r['run_id']} çıkış kodu {rc}: hücreyi yeniden çalıştır (kayıttan devam)\n{lg[-2000:]}"
    ns.check_train_log(open(f"{d}/train_log.txt").read(), MAX_EPOCH, NO_AUG)
    state, rec = store.find(MAX_EPOCH * IPE)
    assert state is not None
    ck = f"{d}/ckpt/{rec['file']}"
    txt, wall_e = ns.stream(["python", "-u", "YOLOX/tools/eval.py", "-f", EXP_FILE, "-c", ck, "-b", "64", "-d", "1",
                             "--conf", "0.001", "--tsize", str(r["res"]), "-expn", f"{r['run_id']}_evalA",
                             "nas_arch", r["config_id"], "nas_seed_init", str(r["seed"]), "output_dir", f"{OUT_ROOT}_eval"],
                            WS, ns.SHOW_EVAL, r["run_id"])
    ev = ns.parse_eval(txt)
    assert ev["map5095"] is not None
    open(f"{d}/eval_A_log.txt.tmp", "w").write(txt); os.replace(f"{d}/eval_A_log.txt.tmp", f"{d}/eval_A_log.txt")
    train_s = round(sum(a.get("wall_s") or 0 for a in st["attempts"]), 1)
    return ns.set_status(DRIVE_RESULTS, r["run_id"], st, status="evaluated", eval_A={**ev, "eval_s": round(wall_e, 1)},
                         ckpt=rec, train_s=train_s, evaluated_utc=ns.now_utc())

for r in RUNS:
    st = train_eval(r)
    print(f"✔ {r['run_id']} {r['config_id']}: A mAP {st['eval_A']['map5095']} · eğitim {st['train_s']} sn", flush=True)

In [ ]:
# 10 · Geliştirme özeti (val2017-A): mimari başına ortalama ± std (n=3), eşli fark (final − Nano) — nihai değil
import csv, statistics as stt
rows = []
for r in RUNS:
    st = ns.get_status(DRIVE_RESULTS, r["run_id"])
    if st["status"] == "evaluated":
        rows.append({**r, "map_A": st["eval_A"]["map5095"], "ap50_A": st["eval_A"]["ap50"], "train_s": st["train_s"]})
with open(f"{DRIVE_RESULTS}/results.csv.tmp", "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=["run_id", "arch", "config_id", "seed", "res", "map_A", "ap50_A", "train_s"],
                       extrasaction="ignore"); w.writeheader(); w.writerows(rows)
os.replace(f"{DRIVE_RESULTS}/results.csv.tmp", f"{DRIVE_RESULTS}/results.csv")
by = {}
for x in rows:
    by.setdefault(x["arch"], {})[x["seed"]] = x["map_A"]
summ = {a: {"n": len(v), "mean": round(stt.mean(v.values()), 3), "std": round(stt.stdev(v.values()), 3) if len(v) > 1 else None}
        for a, v in by.items()}
paired = [by["final"][s] - by["nano"][s] for s in SEEDS if s in by.get("final", {}) and s in by.get("nano", {})]
summ["paired_diff_final_minus_nano"] = {"values": paired, "mean": round(stt.mean(paired), 3) if paired else None,
                                        "note": "val2017-A (geliştirme); nihai rapor mühürlü kümede"}
ns.write_json(f"{DRIVE_RESULTS}/{ROUND}_summary.json", {**summ, "rows": rows, "written_utc": ns.now_utc()})
print(json.dumps(summ, indent=1))